testing the APIs

In [5]:
import os
import pandas as pd
import requests
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
tmdb_key = os.getenv("TMDB_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

def test_tmdb():
    url = f"https://api.themoviedb.org/3/search/tv?api_key={tmdb_key}&query=Dark"
    res = requests.get(url).json()
    if 'results' in res and len(res['results']) > 0:
        print(f"Connected to TMDB! Series found: {res['results'][0]['name']}")
        return res['results'][0]['overview']
    else:
        print("Problem connecting to TMDB")
        return None

def test_claude(text_to_analyze):
    client = Anthropic(api_key=anthropic_key)
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[
            {"role": "user", "content": f"In one sentence, what is the main theme of the series based on the following synopsis: {text_to_analyze}"}
        ]
    )
    print(f"Claude connected! His insight: {message.content[0].text}")

overview = test_tmdb()
if overview:
    test_claude(overview)

Connected to TMDB! Series found: Dark
Claude connected! His insight: The main theme is how a child's disappearance triggers a complex investigation that reveals interconnected secrets and conspiracies involving multiple families across decades.


cleaning the data I got from Netflix

In [ ]:
import time

load_dotenv()
tmdb_key = os.getenv("TMDB_API_KEY")

df = pd.read_csv('ViewingActivity.csv')

# Filter for my profile
df = df[df['Profile Name'] == 'עומרי']

def duration_to_minutes(duration_str):
    try:
        h, m, s = map(int, duration_str.split(':'))
        return h * 60 + m + s / 60
    except:
        return 0

df['Duration_Min'] = df['Duration'].apply(duration_to_minutes)

# Filter the short content (Trailers, clips, or misclicks)
# Keeping only views longer than 3 minutes
df = df[df['Duration_Min'] >= 3.0]

def get_clean_title(title):
    # Netflix titles look like "Show Name: Season : Episode"
    # We want the "Show Name"
    return title.split(':')[0].strip()

df['Clean_Show_Name'] = df['Title'].apply(get_clean_title)
unique_titles = df['Clean_Show_Name'].unique()

print(f"Found {len(unique_titles)} unique shows. Starting enrichment...")


Using TMDB API to enrich the DF
the Rating.df from Netflix contains English names while the ViewingActivity.df contains Hebrew names,
I'll use TMDB to get match the English and Hebrew show names

In [ ]:
import numpy as np

all_results = []

for title in unique_titles:
    print(f"Deep fetching: {title}")
    
    # Search for the ID
    search_url = f"https://api.themoviedb.org/3/search/multi?api_key={tmdb_key}&query={title}&language=en-US"
    
    try:
        search_res = requests.get(search_url).json()
        if search_res.get('results'):
            results_list = search_res['results']
            # Prioritize TV shows, fallback to first result
            match = next((item for item in results_list if item.get('media_type') == 'tv'), results_list[0])
            
            m_id = match['id']
            m_type = match['media_type'] 
            
            # Detailed Fetch
            detail_url = f"https://api.themoviedb.org/3/{m_type}/{m_id}?api_key={tmdb_key}&append_to_response=credits,keywords&language=en-US"
            details = requests.get(detail_url).json()
            
            # Keywords logic
            kw_data = details.get('keywords', {})
            if m_type == 'movie':
                kw_list = kw_data.get('keywords', [])
            else:
                kw_list = kw_data.get('results', [])
            keywords_str = ", ".join([k['name'] for k in kw_list])

            # Runtime logic
            if m_type == 'movie':
                final_runtime = details.get('runtime')
            else:
                # Episode run time is a list, we take the average
                runtimes = details.get('episode_run_time', [])
                final_runtime = np.mean(runtimes) if runtimes else None

            # Get Top 3 Cast members
            cast = [person['name'] for person in details.get('credits', {}).get('cast', [])[:3]]

            all_results.append({
                'Hebrew_Input': title,
                'English_Name': details.get('name') or details.get('title'),
                'Media_Type': m_type,
                'User_Score': details.get('vote_average') * 10 if details.get('vote_average') else 0,
                'Overview': details.get('overview'),
                'Keywords': keywords_str,
                'Runtime_Avg': final_runtime,
                'Genres': ", ".join([g['name'] for g in details.get('genres', [])]),
                'Cast': ", ".join(cast),
                'Release_Date': details.get('first_air_date') if m_type == 'tv' else details.get('release_date'),
                'Status': details.get('status')
            })
    except Exception as e:
        print(f"Error for {title}: {e}")
    
    time.sleep(0.1)

enriched_df = pd.DataFrame(all_results)
# Save to CSV so I have it backed up
enriched_df.to_csv('TMDB_data.csv', index=False)
display(enriched_df.head(10))

Deep fetching: הדינוזאורים
Deep fetching: ברידג'רטון
Deep fetching: התבגרות
Deep fetching: אקווינוקס
Deep fetching: המפתחות של משפחת לוק
Deep fetching: ארכיון 81
Deep fetching: האהבה היא עיוורת
Deep fetching: נפשות תאומות
Deep fetching: סימן חיים
Deep fetching: זרה
Deep fetching: הפלא
Deep fetching: פרומתיאוס
Deep fetching: מי מכיר אותך?
Deep fetching: שלו ושלה
Deep fetching: הרלן קובן
Deep fetching: זמן הזבובים
Deep fetching: דברים מוזרים
Deep fetching: דאהמר
Deep fetching: האדם נגד התינוק
Deep fetching: משחקים גרייסי דרלינג
Deep fetching: אבסנטיה
Deep fetching: החיה שבתוכי
Deep fetching: נרו
Deep fetching: ספרטקוס
Deep fetching: ויקינגים
Deep fetching: המכשף
Deep fetching: תיקון מידות
Deep fetching: מפלצת
Deep fetching: המפלצת מפירנצה
Deep fetching: אף אחד לא ראה שעזבנו
Deep fetching: הנוסעים בזמן
Deep fetching: ונסדיי
Deep fetching: בן ערובה
Deep fetching: בעיית שלושת הגופים
Deep fetching: ‎1899‎
Deep fetching: מראה שחורה
Deep fetching: מבפנים
Deep fetching: הניצולים
Deep fetching: 

,Hebrew_Input,English_Name,Media_Type,User_Score,Overview,Keywords,Runtime_Avg,Genres,Cast,Release_Date,Status
0,הדינוזאורים,The Dinosaurs,tv,79.42,"From the first dinosaurs to the last, this epi...","natural disaster, mass extinction, evolution, ...",NaN,"Documentary, Animation",Morgan Freeman,2026-03-06,Ended
1,ברידג'רטון,Bridgerton,tv,80.63,"Wealth, lust, and betrayal set in the backdrop...","based on novel or book, society, marriage, lov...",NaN,Drama,"Ruth Gemmell, Luke Thompson, Yerin Ha",2020-12-25,Returning Series
2,התבגרות,Adolescence,tv,78.52,When a 13-year-old is accused of the murder of...,"england, sexism, violence against women, patri...",NaN,"Drama, Crime",,2025-03-13,Ended
3,אקווינוקס,Equinox,tv,71.00,Haunted by visions after her sister vanished w...,"denmark, island, trauma, investigation, forest...",47.0,"Sci-Fi & Fantasy, Drama","Ask Truelsen, Danica Ćurčić, Lars Brygmann",2020-12-30,Ended
4,המפתחות של משפחת לוק,Locke & Key,tv,77.47,"After their dad's murder, three siblings move ...","small town, magic, based on comic, supernatura...",NaN,"Sci-Fi & Fantasy, Drama, Mystery","Darby Stanchfield, Connor Jessup, Emilia Jones",2020-02-07,Ended
5,ארכיון 81,Archive 81,tv,72.10,An archivist takes a job restoring damaged vid...,"camera, comet, supernatural, cult, paranormal,...",NaN,"Drama, Mystery, Sci-Fi & Fantasy","Mamoudou Athie, Dina Shihabi, Evan Jonigkeit",2022-01-14,Canceled
6,האהבה היא עיוורת,Love Is Blind: Poland,tv,100.00,"In this unique dating experiment, Polish singl...","blind date, love, dating show, single ladies, ...",60.0,Reality,"Zofia Zborowska-Wrona, Andrzej Wrona",2026-05-06,Returning Series
7,נפשות תאומות,Soulmates,tv,61.00,"Fifteen years into the future, science has mad...","soulmates, anthology, romance, near future",NaN,"Sci-Fi & Fantasy, Crime, Drama",,2020-10-05,Canceled
8,סימן חיים,Give a Life Sign,movie,0.00,Tamir Gal's Hello Sigal's sequel.,,90.0,,,2002-10-06,Released
9,זרה,Outlander,tv,81.79,"The story of Claire Randall, a married combat ...","scotland, based on novel or book, magic, time ...",NaN,"Drama, Sci-Fi & Fantasy","Caitríona Balfe, Sam Heughan, Sophie Skelton",2014-08-09,Returning Series


Now I'll clean the ViewingActivity and Ratings data, then mergging it all togther with TMDB_data
also need to add more data from TMDB, for example num of episodes

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv()
tmdb_key = os.getenv("TMDB_API_KEY")

def duration_to_minutes(duration_str):
    try:
        h, m, s = map(int, duration_str.split(':'))
        return h * 60 + m + s / 60
    except:
        return 0

# ── Load existing data ────────────────────────────────────────────────────────

viewing_df = pd.read_csv('ViewingActivity.csv')
ratings_df = pd.read_csv('Ratings.csv')
tmdb_df    = pd.read_csv('TMDB_data.csv')

viewing_df = viewing_df[viewing_df['Profile Name'] == 'עומרי'].copy()
viewing_df['Duration_Min']    = viewing_df['Duration'].apply(duration_to_minutes)
viewing_df = viewing_df[viewing_df['Duration_Min'] >= 3.0]
viewing_df['Clean_Show_Name'] = viewing_df['Title'].apply(lambda t: t.split(':')[0].strip())
viewing_df['Start Time']      = pd.to_datetime(viewing_df['Start Time'], utc=True)
viewing_df['Start_IL']        = viewing_df['Start Time'].dt.tz_convert('Asia/Jerusalem')

ratings_df = ratings_df[ratings_df['Profile Name'] == 'עומרי'].copy()
ratings_clean = (
    ratings_df
    .groupby('Title Name', as_index=False)['Thumbs Value']
    .max()
    .rename(columns={'Title Name': 'Clean_Show_Name', 'Thumbs Value': 'Rating'})
)

# ── Fetch missing fields from TMDB ──────────────────────────────────────

existing_titles = tmdb_df['Hebrew_Input'].tolist()

new_fields = []
for i, row in tmdb_df.iterrows():
    title  = row['Hebrew_Input']
    m_id   = row.get('TMDB_ID')
    m_type = row.get('Media_Type', 'tv')

    # If TMDB_ID is missing, re-search
    if pd.isna(m_id):
        search_url = f"https://api.themoviedb.org/3/search/multi?api_key={tmdb_key}&query={title}&language=en-US"
        try:
            res = requests.get(search_url, timeout=10).json()
            results = res.get('results', [])
            if not results:
                new_fields.append({'Hebrew_Input': title})
                continue
            match  = next((r for r in results if r.get('media_type') == 'tv'), results[0])
            m_id   = match['id']
            m_type = match.get('media_type', 'tv')
        except Exception as e:
            print(f"  [search error] {title}: {e}")
            new_fields.append({'Hebrew_Input': title})
            continue

    detail_url = f"https://api.themoviedb.org/3/{m_type}/{int(m_id)}?api_key={tmdb_key}&language=en-US"
    try:
        details = requests.get(detail_url, timeout=10).json()
    except Exception as e:
        print(f"  [detail error] {title}: {e}")
        new_fields.append({'Hebrew_Input': title})
        continue

    seasons_data = [
        s for s in details.get('seasons', [])
        if s.get('season_number', 0) > 0
    ]
    total_episodes_calc = sum(s.get('episode_count', 0) for s in seasons_data)

    new_fields.append({
        'Hebrew_Input':            title,
        'Num_Seasons':             details.get('number_of_seasons'),
        'Num_Episodes':            details.get('number_of_episodes'),
        'Num_Episodes_Calculated': total_episodes_calc,
        'Popularity':              details.get('popularity'),
        'Networks':                ", ".join(n['name'] for n in details.get('networks', [])),
        'Origin_Country':          ", ".join(details.get('origin_country', [])),
    })

    time.sleep(0.15)

new_fields_df = pd.DataFrame(new_fields)
tmdb_df = tmdb_df.merge(new_fields_df, on='Hebrew_Input', how='left')
tmdb_df.to_csv('TMDB_data.csv', index=False)
print(f"\nTMDB_data.csv updated with new fields.")

# ── Merge all tables ──────────────────────────────────────────────────────────

merged = viewing_df.merge(tmdb_df, left_on='Clean_Show_Name', right_on='Hebrew_Input', how='left')
merged = merged.merge(ratings_clean, on='Clean_Show_Name', how='left')
merged['Rating'] = merged['Rating'].fillna(0).astype(int)

print(f"Merged shape: {merged.shape}")
print(f"\nRating distribution:\n{merged.drop_duplicates('Clean_Show_Name')['Rating'].value_counts().sort_index()}")
display(merged.head())

  [1/242] הדינוזאורים → 4 eps
  [2/242] ברידג'רטון → 32 eps
  [3/242] התבגרות → 4 eps
  [4/242] אקווינוקס → 6 eps
  [5/242] המפתחות של משפחת לוק → 28 eps
  [6/242] ארכיון 81 → 8 eps
  [7/242] האהבה היא עיוורת → 11 eps
  [8/242] נפשות תאומות → 6 eps
  [9/242] סימן חיים → None eps
  [10/242] זרה → 101 eps
  [11/242] הפלא → 99 eps
  [12/242] פרומתיאוס → None eps
  [13/242] מי מכיר אותך? → 6 eps
  [14/242] שלו ושלה → 6 eps
  [15/242] הרלן קובן → 8 eps
  [16/242] זמן הזבובים → 6 eps
  [17/242] דברים מוזרים → 42 eps
  [18/242] דאהמר → 10 eps
  [19/242] האדם נגד התינוק → 4 eps
  [20/242] משחקים גרייסי דרלינג → 6 eps
  [21/242] אבסנטיה → 30 eps
  [22/242] החיה שבתוכי → 8 eps
  [23/242] נרו → 8 eps
  [24/242] ספרטקוס → 33 eps
  [25/242] ויקינגים → 89 eps
  [26/242] המכשף → 32 eps
  [27/242] תיקון מידות → 8 eps
  [28/242] מפלצת → 74 eps
  [29/242] המפלצת מפירנצה → 4 eps
  [30/242] אף אחד לא ראה שעזבנו → 5 eps
  [31/242] הנוסעים בזמן → 34 eps
  [32/242] ונסדיי → 16 eps
  [33/242] בן ערובה → 6 eps

,Duration,Start Time,Profile Name,Country,Bookmark,Latest Bookmark,Supplemental Video Type,Attributes,Device Type,Title,...,Cast,Release_Date,Status,Num_Seasons,Num_Episodes,Num_Episodes_Calculated,Popularity,Networks,Origin_Country,Rating
0,00:04:57,2026-03-15 16:53:12+00:00,עומרי,IL (Israel),00:07:25,00:07:25,NaN,NaN,Chrome PC (Cadmium),הדינוזאורים: עונה 1: ‏עליית השושלת (פרק 1),...,Morgan Freeman,2026-03-06,Ended,1.0,4.0,4.0,6.4336,Netflix,"GB, US",0
1,00:10:51,2026-03-13 12:36:04+00:00,עומרי,IL (Israel),00:12:58,00:12:58,NaN,NaN,Chrome PC (Cadmium),ברידג'רטון: עונה 2: ‏זמזומים טורדניים (פרק 3),...,"Ruth Gemmell, Luke Thompson, Yerin Ha",2020-12-25,Returning Series,5.0,32.0,32.0,62.0349,Netflix,US,0
2,00:18:01,2026-03-13 11:38:37+00:00,עומרי,IL (Israel),00:50:00,00:50:00,NaN,NaN,Chrome PC (Cadmium),ברידג'רטון: עונה 2: ‏על קו הזינוק (פרק 2),...,"Ruth Gemmell, Luke Thompson, Yerin Ha",2020-12-25,Returning Series,5.0,32.0,32.0,62.0349,Netflix,US,0
3,00:14:14,2026-03-12 16:40:49+00:00,עומרי,IL (Israel),00:32:30,Not latest view,NaN,NaN,Chrome PC (Cadmium),ברידג'רטון: עונה 2: ‏על קו הזינוק (פרק 2),...,"Ruth Gemmell, Luke Thompson, Yerin Ha",2020-12-25,Returning Series,5.0,32.0,32.0,62.0349,Netflix,US,0
4,00:26:58,2026-03-12 13:15:54+00:00,עומרי,IL (Israel),00:34:17,00:34:17,NaN,NaN,Chrome PC (Cadmium),התבגרות: סדרה מוגבלת: ‏פרק 1 (פרק 1),...,NaN,2025-03-13,Ended,1.0,4.0,4.0,18.0411,Netflix,"GB, US",0


In [ ]:
# Map English_Name -> Clean_Show_Name (Hebrew) via tmdb_df to align ratings with the enriched data
english_to_hebrew = tmdb_df.set_index('English_Name')['Hebrew_Input'].to_dict()

ratings_clean = (
    ratings_df
    .groupby('Title Name', as_index=False)['Thumbs Value']
    .max()
    .rename(columns={'Thumbs Value': 'Rating'})
)
ratings_clean['Clean_Show_Name'] = ratings_clean['Title Name'].map(english_to_hebrew)

# Check how many matched
matched   = ratings_clean['Clean_Show_Name'].notna().sum()
unmatched = ratings_clean['Clean_Show_Name'].isna().sum()
print(f"Matched: {matched}, Unmatched: {unmatched}")
print("\nUnmatched titles:")
print(ratings_clean[ratings_clean['Clean_Show_Name'].isna()]['Title Name'].tolist())

Matched: 37, Unmatched: 9

Unmatched titles:
['Another Life', 'DAHMER', "Evil Genius: the True Story of America's Most Diabolical Bank Heist", 'Flatliners', 'Grimm', 'Intrusion', 'Sex/Life', 'The Innocents', 'The Originals']


I'll manualy add translation to the unmatched titles

In [28]:
manual_map = {
    'Another Life':           'חיים אחרים',
    'DAHMER':                 'דאהמר',
    'Evil Genius: the True Story of America\'s Most Diabolical Bank Heist': 'גאון מרושע',
    'Flatliners':             'קו הדממה',
    'Grimm':                  'גרים',
    'Intrusion':              'פריצה',
    'Sex/Life':               None,   # not in my viewing history
    'The Innocents':          'התמימים',
    'The Originals':          'המקוריים',
}

# Fill unmatched using manual map
ratings_clean['Clean_Show_Name'] = ratings_clean.apply(
    lambda row: manual_map.get(row['Title Name'], row['Clean_Show_Name'])
    if pd.isna(row['Clean_Show_Name']) else row['Clean_Show_Name'],
    axis=1
)

# Drop entries that have no match at all
ratings_clean = ratings_clean.dropna(subset=['Clean_Show_Name'])

print(f"Final ratings count: {len(ratings_clean)}")
print(ratings_clean[['Title Name', 'Clean_Show_Name', 'Rating']])

Final ratings count: 45
                                           Title Name  \
0                                                1899   
1                                      3 Body Problem   
2                                       6 Underground   
3                                        Another Life   
4                                               Arrow   
5                                       Baby Reindeer   
6                                          Biohackers   
7                                              DAHMER   
8                                                Dark   
9                                          Dear Child   
10  Evil Genius: the True Story of America's Most ...   
11                                         Flatliners   
12                                          Fractured   
13                                              Grimm   
14                                              Helix   
15                                     Into the Night   
16     

In [29]:
merged = viewing_df.merge(tmdb_df, left_on='Clean_Show_Name', right_on='Hebrew_Input', how='left')
merged = merged.merge(ratings_clean[['Clean_Show_Name', 'Rating']], on='Clean_Show_Name', how='left')
merged['Rating'] = merged['Rating'].fillna(0).astype(int)

print(f"Rating distribution:\n{merged.drop_duplicates('Clean_Show_Name')['Rating'].value_counts().sort_index()}")

Rating distribution:
Rating
0    204
1      2
2     23
3     17
Name: count, dtype: int64


Calculating stats such as Completion ratio, Binge score, Engagement score

In [ ]:
# ── Informed caps from distribution analysis below ──────────────────────────────────
CAP_AVG_DAILY   = 4.0    # P90
CAP_MAX_EPS_DAY = 9.0    # P95
CAP_BINGE_DAYS  = 8.0    # P95

show_stats = []

for show, group in merged.groupby('Clean_Show_Name'):

    total_minutes_watched = group['Duration_Min'].sum()
    num_sessions          = len(group)
    rating                = group['Rating'].iloc[0]

    num_episodes = group['Num_Episodes'].iloc[0]
    num_seasons  = group['Num_Seasons'].iloc[0]
    runtime_avg  = group['Runtime_Avg'].iloc[0]
    genres       = group['Genres'].iloc[0]
    keywords     = group['Keywords'].iloc[0]
    user_score   = group['User_Score'].iloc[0]
    status       = group['Status'].iloc[0]
    english_name = group['English_Name'].iloc[0]

    # Estimated episodes watched (separate from sessions)
    if pd.notna(runtime_avg) and runtime_avg > 0:
        est_eps_watched = total_minutes_watched / runtime_avg
    elif pd.notna(num_episodes):
        est_eps_watched = total_minutes_watched / 40
    else:
        est_eps_watched = np.nan

    # ── Completion ratio ──────────────────────────────────────────────────────
    if pd.notna(num_episodes) and pd.notna(runtime_avg) and runtime_avg > 0:
        estimated_total_min = num_episodes * runtime_avg
        raw_completion      = total_minutes_watched / estimated_total_min
    elif pd.notna(num_episodes) and num_episodes > 0:
        estimated_total_min = num_episodes * 40
        raw_completion      = total_minutes_watched / estimated_total_min
    else:
        raw_completion = np.nan

    completion_ratio = min(raw_completion, 1.0) if pd.notna(raw_completion) else np.nan
    # raw_completion > 1.0 signals re-watches — kept separately
    completion_fill  = completion_ratio if pd.notna(completion_ratio) else 0.3

    # ── Binge & pace features ─────────────────────────────────────────────────
    dates            = pd.to_datetime(group['Start_IL']).dt.date
    episodes_per_day = group.groupby(dates)['Duration_Min'].count()

    max_eps_in_day    = episodes_per_day.max()
    avg_daily_eps     = episodes_per_day.mean()
    binge_days        = (episodes_per_day >= 3).sum()
    date_range        = (dates.max() - dates.min()).days + 1

    # Each component normalized independently using data-driven caps
    max_eps_norm  = min(max_eps_in_day / CAP_MAX_EPS_DAY, 1.0)   # intensity
    binge_norm    = min(binge_days     / CAP_BINGE_DAYS,  1.0)   # sustained binge
    pace_norm     = min(avg_daily_eps  / CAP_AVG_DAILY,   1.0)   # overall pace

    # binge_score: intensity-weighted combination (no raw multiplication)
    binge_score = (max_eps_norm * 0.6) + (binge_norm * 0.4)

    # ── Engagement score ──────────────────────────────────────────────────────
    # rating=1 -> 0.0, rating=2 -> 0.5, rating=3 -> 1.0
    rating_norm = (rating - 1) / 2.0

    if rating > 0:
        # Explicit rating available: trust it most
        engagement_score = (
            0.40 * rating_norm    +
            0.30 * completion_fill +
            0.20 * binge_score    +
            0.10 * pace_norm
        )
    else:
        # No rating: behavioral signals only
        engagement_score = (
            0.50 * completion_fill +
            0.30 * binge_score    +
            0.20 * pace_norm
        )

    show_stats.append({
        'show':              show,
        'english_name':      english_name,
        'genres':            genres,
        'keywords':          keywords,
        'rating':            rating,
        'user_score':        user_score,
        'status':            status,
        'num_episodes':      num_episodes,
        'num_seasons':       num_seasons,
        'runtime_avg':       runtime_avg,
        'total_minutes':     total_minutes_watched,
        'num_sessions':      num_sessions,
        'est_eps_watched':   round(est_eps_watched, 1) if pd.notna(est_eps_watched) else np.nan,
        'raw_completion':    round(raw_completion, 4)  if pd.notna(raw_completion)  else np.nan,
        'completion_ratio':  round(completion_ratio, 4) if pd.notna(completion_ratio) else np.nan,
        'max_eps_in_day':    max_eps_in_day,
        'avg_daily_eps':     round(avg_daily_eps, 3),
        'binge_days':        binge_days,
        'binge_score':       round(binge_score, 4),
        'pace_norm':         round(pace_norm, 4),
        'date_range_days':   date_range,
        'engagement_score':  round(engagement_score, 4),
    })

show_df = pd.DataFrame(show_stats).sort_values('engagement_score', ascending=False).reset_index(drop=True)

print(f"show_df shape: {show_df.shape}")
print(f"\nTop 10 by engagement:")
display(show_df[['show', 'rating', 'completion_ratio', 'binge_score', 'engagement_score']].head(10))
print(f"\nBottom 10 by engagement:")
display(show_df[['show', 'rating', 'completion_ratio', 'binge_score', 'engagement_score']].tail(10))

show_df shape: (246, 22)

Top 10 by engagement:


,show,rating,completion_ratio,binge_score,engagement_score
0,המכשף,3,1.0,1.0000,1.0000
1,מיינד האנטר,3,1.0,1.0000,1.0000
2,בעיית שלושת הגופים,3,1.0,0.7500,0.9500
3,הנוסעים בזמן,0,1.0,1.0000,0.9431
4,תישאר קרוב,0,1.0,0.7000,0.9100
5,החיה שבתוכי,3,1.0,0.5500,0.9100
6,אפל,3,1.0,0.8000,0.9057
7,האהבה היא עיוורת,0,1.0,1.0000,0.9010
8,נפילת בית אשר,3,1.0,0.5333,0.9004
9,חינוך מיני,0,1.0,0.9000,0.8986



Bottom 10 by engagement:


,show,rating,completion_ratio,binge_score,engagement_score
236,להמציא את אנה,0,0.0115,0.0667,0.0757
237,האחות,0,0.0085,0.0667,0.0743
238,רכבת הקרח,0,0.0066,0.0667,0.0733
239,ספרטקוס,0,0.0055,0.0667,0.0727
240,מסע בין כוכבים,0,0.0049,0.0667,0.0724
241,"The Circle‎ ‏- ארה""ב",0,0.0035,0.0667,0.0718
242,שלדון הצעיר,0,0.0037,0.0667,0.0718
243,ג'יני וג'ורג'יה,0,0.0031,0.0667,0.0716
244,חברים,0,0.0025,0.0667,0.0713
245,ואן הלסינג,0,0.0016,0.0667,0.0708


In [31]:
print("avg_daily_episodes distribution:")
print(show_df['avg_daily_episodes'].describe(percentiles=[.5, .75, .90, .95]))

print("\nmax_eps_in_day distribution:")
print(show_df['max_eps_in_day'].describe(percentiles=[.5, .75, .90, .95]))

print("\nbinge_days distribution:")
print(show_df['binge_days'].describe(percentiles=[.5, .75, .90, .95]))

print("\ncompletion_ratio distribution:")
print(show_df['completion_ratio'].describe(percentiles=[.25, .5, .75, .90]))

avg_daily_episodes distribution:
count    246.000000
mean       2.109748
std        1.183996
min        1.000000
50%        1.814141
75%        2.770833
90%        3.732143
95%        4.500000
max        7.000000
Name: avg_daily_episodes, dtype: float64

max_eps_in_day distribution:
count    246.000000
mean       3.434959
std        2.468131
min        1.000000
50%        3.000000
75%        5.000000
90%        7.000000
95%        9.000000
max       11.000000
Name: max_eps_in_day, dtype: float64

binge_days distribution:
count    246.000000
mean       2.012195
std        4.335409
min        0.000000
50%        1.000000
75%        2.000000
90%        4.500000
95%        8.000000
max       40.000000
Name: binge_days, dtype: float64

completion_ratio distribution:
count    201.000000
mean       0.504672
std        0.401879
min        0.001569
25%        0.107196
50%        0.423438
75%        1.000000
90%        1.000000
max        1.000000
Name: completion_ratio, dtype: float64


Calculating geners vectors based on the engagement score

In [35]:
from sklearn.preprocessing import MultiLabelBinarizer

# ── Genre vector ──────────────────────────────────────────────────────────────

# Parse genres from string to list, filter shows with no genre data
show_df['genres_list'] = show_df['genres'].apply(
    lambda x: [g.strip() for g in x.split(',')] if pd.notna(x) and x != '' else []
)

mlb_genre = MultiLabelBinarizer()
genre_matrix = mlb_genre.fit_transform(show_df['genres_list'])
genre_df = pd.DataFrame(genre_matrix, columns=mlb_genre.classes_, index=show_df.index)

print(f"Genre matrix: {genre_df.shape}")
print(f"Genres found: {list(mlb_genre.classes_)}")

# ── Weighted preference vector ────────────────────────────────────────────────

weights = show_df['engagement_score'].values.reshape(-1, 1)
weighted_genre_matrix = genre_matrix * weights
preference_vector_genre = weighted_genre_matrix.sum(axis=0) / weights.sum()

preference_genre_df = pd.Series(
    preference_vector_genre,
    index=mlb_genre.classes_
).sort_values(ascending=False)

print(f"\nYour genre preference vector:")
display(preference_genre_df.round(4))

Genre matrix: (246, 21)
Genres found: ['Action', 'Action & Adventure', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Horror', 'Kids', 'Music', 'Mystery', 'Reality', 'Romance', 'Sci-Fi & Fantasy', 'Science Fiction', 'Soap', 'Thriller', 'War & Politics']

Your genre preference vector:


Drama                 0.7230
Mystery               0.3202
Crime                 0.3193
Sci-Fi & Fantasy      0.3108
Action & Adventure    0.1134
Comedy                0.1130
Documentary           0.0994
Science Fiction       0.0386
Reality               0.0379
Thriller              0.0342
Action                0.0200
Animation             0.0173
Horror                0.0165
Family                0.0156
War & Politics        0.0156
Adventure             0.0088
Kids                  0.0073
Fantasy               0.0042
Music                 0.0036
Romance               0.0027
Soap                  0.0019
dtype: float64

In [36]:
genre_merge_map = {
    'Science Fiction': 'Sci-Fi & Fantasy',
    'Fantasy':         'Sci-Fi & Fantasy',
    'Action':          'Action & Adventure',
    'Adventure':       'Action & Adventure',
}

show_df['genres_list'] = show_df['genres_list'].apply(
    lambda genres: list(set(genre_merge_map.get(g, g) for g in genres))
)

# Rebuild matrix with merged genres
mlb_genre = MultiLabelBinarizer()
genre_matrix = mlb_genre.fit_transform(show_df['genres_list'])
genre_df = pd.DataFrame(genre_matrix, columns=mlb_genre.classes_, index=show_df.index)

weights = show_df['engagement_score'].values.reshape(-1, 1)
preference_vector_genre = (genre_matrix * weights).sum(axis=0) / weights.sum()

preference_genre_df = pd.Series(
    preference_vector_genre,
    index=mlb_genre.classes_
).sort_values(ascending=False)

print("Genre preference vector (merged):")
display(preference_genre_df.round(4))

Genre preference vector (merged):


Drama                 0.7230
Sci-Fi & Fantasy      0.3515
Mystery               0.3202
Crime                 0.3193
Action & Adventure    0.1380
Comedy                0.1130
Documentary           0.0994
Reality               0.0379
Thriller              0.0342
Animation             0.0173
Horror                0.0165
Family                0.0156
War & Politics        0.0156
Kids                  0.0073
Music                 0.0036
Romance               0.0027
Soap                  0.0019
dtype: float64

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Keyword preference vector ─────────────────────────────────────────────────

# Filter shows with keywords
kw_df = show_df[show_df['keywords'].notna() & (show_df['keywords'] != '')].copy()
print(f"Shows with keywords: {len(kw_df)} / {len(show_df)}")

# TF-IDF: gives more weight to specific keywords, less to common ones like "based on novel"
tfidf = TfidfVectorizer(max_features=100, token_pattern=r"[^,]+")
tfidf_matrix = tfidf.fit_transform(kw_df['keywords'])

# Weighted by engagement_score
kw_weights = kw_df['engagement_score'].values.reshape(-1, 1)
weighted_tfidf = tfidf_matrix.toarray() * kw_weights
preference_vector_kw = weighted_tfidf.sum(axis=0) / kw_weights.sum()

preference_kw_df = pd.Series(
    preference_vector_kw,
    index=tfidf.get_feature_names_out()
).sort_values(ascending=False)

print("\nTop 30 keywords in your preference vector:")
display(preference_kw_df.head(30).round(4))
print("\nBottom 10 keywords (things you avoid):")
display(preference_kw_df.tail(10).round(4))

Shows with keywords: 222 / 246

Top 30 keywords in your preference vector:


 miniseries                0.0843
based on novel or book     0.0561
 thriller                  0.0553
 murder                    0.0437
 based on novel or book    0.0398
 based on comic            0.0342
miniseries                 0.0340
 investigation             0.0304
 lgbt                      0.0280
 mysterious                0.0270
 small town                0.0267
 dark                      0.0251
 suspenseful               0.0248
 love                      0.0247
 true crime                0.0245
 california                0.0240
 intimate                  0.0233
 horror                    0.0233
 supernatural              0.0231
 science fiction           0.0227
 dystopia                  0.0226
 dark comedy               0.0226
 complex                   0.0220
 detective                 0.0215
 serial killer             0.0209
 magic                     0.0208
 friendship                0.0202
 missing person            0.0198
 psychological thriller    0.0197
 disappearance


Bottom 10 keywords (things you avoid):


missing child     0.0062
action            0.0061
based on movie    0.0058
informative       0.0057
meditative        0.0043
hilarious         0.0042
sentimental       0.0038
optimistic        0.0032
based on manga    0.0032
anime             0.0032
dtype: float64

In [39]:
# Fix whitespace in keywords before running TF-IDF
kw_df = show_df[show_df['keywords'].notna() & (show_df['keywords'] != '')].copy()

# Strip whitespace from each keyword at the tokenizer level
def tokenize_keywords(text):
    return [k.strip() for k in text.split(',') if k.strip()]

tfidf = TfidfVectorizer(max_features=100, analyzer='word', tokenizer=tokenize_keywords)
tfidf_matrix = tfidf.fit_transform(kw_df['keywords'])

kw_weights = kw_df['engagement_score'].values.reshape(-1, 1)
weighted_tfidf = tfidf_matrix.toarray() * kw_weights
preference_vector_kw = weighted_tfidf.sum(axis=0) / kw_weights.sum()

preference_kw_df = pd.Series(
    preference_vector_kw,
    index=tfidf.get_feature_names_out()
).sort_values(ascending=False)

print("Top 30 keywords:")
display(preference_kw_df.head(30).round(4))
print("\nBottom 10 keywords:")
display(preference_kw_df.tail(10).round(4))

Top 30 keywords:


c:\Users\omris\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


miniseries                0.1094
based on novel or book    0.0789
thriller                  0.0520
murder                    0.0432
small town                0.0353
based on comic            0.0336
detective                 0.0308
based on true story       0.0302
investigation             0.0293
time travel               0.0288
love                      0.0273
lgbt                      0.0269
dystopia                  0.0264
dark                      0.0263
true crime                0.0252
suspenseful               0.0251
supernatural              0.0244
dark comedy               0.0244
mysterious                0.0242
intimate                  0.0241
horror                    0.0238
serial killer             0.0238
superhero                 0.0222
complex                   0.0217
magic                     0.0214
science fiction           0.0208
friendship                0.0207
california                0.0198
1960s                     0.0194
psychological thriller    0.0193
dtype: flo


Bottom 10 keywords:


meditative              0.0067
android                 0.0064
family relationships    0.0063
bullying                0.0062
teen drama              0.0060
action                  0.0055
hospital                0.0053
hilarious               0.0041
based on manga          0.0038
optimistic              0.0034
dtype: float64

uses the Anthropic API to analyze TV show overviews and automatically generate structured, 1-to-5 thematic profiles (such as pacing, darkness, and complexity) for each show using Claude. 

In [42]:
from anthropic import Anthropic

client = Anthropic()

import json
import re
import time

def get_thematic_profile(overview, keywords, genres):
    prompt = f"""Analyze this TV show and return a JSON profile. Be precise and data-driven.

Show data:
- Genres: {genres}  
- Keywords: {keywords}
- Overview: {overview}

Return ONLY this JSON structure with no other text:
{{
  "pacing": <1-5, where 1=very slow burn, 5=very fast paced>,
  "darkness": <1-5, where 1=light/uplifting, 5=very dark/heavy>,
  "complexity": <1-5, where 1=simple/linear, 5=mind-bending/complex>,
  "emotional_intensity": <1-5, where 1=detached, 5=very emotionally intense>,
  "mystery_level": <1-5, where 1=no mystery, 5=central mystery throughout>,
  "realism": <1-5, where 1=pure fantasy, 5=very grounded/realistic>,
  "tone_tags": <array of 2-4 single words from: ["tense","humorous","romantic","disturbing","inspiring","nostalgic","philosophical","suspenseful","whimsical","gritty"]>
}}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text
    # Strip ```json ... ``` wrapper if present
    cleaned = re.sub(r'^```json\s*|\s*```$', '', raw.strip())
    try:
        return json.loads(cleaned)
    except:
        print(f"  [parse error] raw: {repr(raw)}")
        return None

# ── Run on all shows ──────────────────────────────────────────────────────────

shows_for_llm = show_df.merge(
    tmdb_df[['Hebrew_Input', 'Overview']],
    left_on='show', right_on='Hebrew_Input', how='left'
)

thematic_profiles = {}
failed = []

for i, row in shows_for_llm.iterrows():
    if pd.notna(row.get('Overview')) and row['Overview'] != '':
        profile = get_thematic_profile(row['Overview'], row['keywords'], row['genres'])
        if profile:
            thematic_profiles[row['show']] = profile
        else:
            failed.append(row['show'])
    else:
        failed.append(row['show'])
    time.sleep(0.1)

print(f"Success: {len(thematic_profiles)} / {len(shows_for_llm)}")
print(f"Failed:  {len(failed)} → {failed[:10]}")

# ── Add numeric columns to show_df ───────────────────────────────────────────

thematic_cols = ['pacing', 'darkness', 'complexity', 'emotional_intensity', 'mystery_level', 'realism']

for col in thematic_cols:
    show_df[col] = show_df['show'].map(
        lambda s: thematic_profiles[s][col] if s in thematic_profiles else np.nan
    )

show_df['tone_tags'] = show_df['show'].map(
    lambda s: thematic_profiles[s]['tone_tags'] if s in thematic_profiles else []
)

print(f"\nThematic profile averages (unweighted):")
display(show_df[thematic_cols].mean().round(2))

print(f"\nWeighted by engagement_score:")
weights = show_df['engagement_score']
weighted_means = {
    col: (show_df[col] * weights).sum() / weights.sum()
    for col in thematic_cols
}
display(pd.Series(weighted_means).round(2))

Success: 239 / 246
Failed:  7 → ['גרים', 'חיים אחרים', 'המִסדר', 'מגרש המשחקים האנושי', 'ניל בלומקאמפ ו\u202bאולפני Oats מציגים', 'ספר לי מי אני', 'הצלה']

Thematic profile averages (unweighted):


pacing                 3.54
darkness               3.63
complexity             3.44
emotional_intensity    3.94
mystery_level          3.78
realism                3.20
dtype: float64


Weighted by engagement_score:


pacing                 3.45
darkness               3.65
complexity             3.43
emotional_intensity    3.90
mystery_level          3.84
realism                3.12
dtype: float64

Modeling the data
This script trains a Linear Regression model to validate whether the engineered engagement_score is a reliable target metric. While including behavioral features creates data leakage for future predictions, it serves here as a validation step to ensure the score strongly correlates with actual viewing habits like bingeing and completion rates

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

# ── Build feature matrix ──────────────────────────────────────────────────────
# Only shows with enough data for modeling
model_df = show_df.dropna(subset=['completion_ratio', 'pacing', 'darkness',
                                   'complexity', 'emotional_intensity',
                                   'mystery_level', 'realism']).copy()

print(f"Shows available for modeling: {len(model_df)}")

# Genre one-hot
mlb = MultiLabelBinarizer()
genre_ohe = mlb.fit_transform(model_df['genres_list'])
genre_cols = [f"genre_{g.replace(' ', '_').replace('&','and').replace('-','_')}"
              for g in mlb.classes_]

# Tone tags one-hot
all_tone_tags = ['tense','humorous','romantic','disturbing',
                 'inspiring','nostalgic','philosophical',
                 'suspenseful','whimsical','gritty']
for tag in all_tone_tags:
    model_df[f'tone_{tag}'] = model_df['tone_tags'].apply(
        lambda x: 1 if isinstance(x, list) and tag in x else 0
    )
tone_cols = [f'tone_{t}' for t in all_tone_tags]

# Assemble full feature matrix
thematic_cols = ['pacing', 'darkness', 'complexity',
                 'emotional_intensity', 'mystery_level', 'realism']
behavioral_cols = ['completion_ratio', 'binge_score', 'pace_norm']

feature_cols = thematic_cols + behavioral_cols + tone_cols
genre_df_ohe = pd.DataFrame(genre_ohe, columns=genre_cols, index=model_df.index)
X = pd.concat([model_df[feature_cols], genre_df_ohe], axis=1)
y = model_df['engagement_score']

print(f"Feature matrix: {X.shape}")

# ── Train/Test split (80/20) ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} shows | Test: {len(X_test)} shows")

# ── Linear Regression ─────────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print(f"\nLinear Regression:")
print(f"  MAE : {mae_lr:.4f}  (avg error in engagement score units)")
print(f"  R²  : {r2_lr:.4f}  (1.0 = perfect, 0 = no better than mean)")

# ── Feature importance ────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature':     X.columns,
    'coefficient': lr.coef_
}).sort_values('coefficient', ascending=False)

print(f"\nTop 10 features that predict HIGH engagement:")
display(coef_df.head(10))
print(f"\nTop 10 features that predict LOW engagement:")
display(coef_df.tail(10))

# ── Predicted vs Actual ───────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'show':       model_df.loc[X_test.index, 'show'],
    'actual':     y_test.values,
    'predicted':  y_pred_lr.round(4),
    'error':      (y_pred_lr - y_test.values).round(4)
}).sort_values('error', key=abs, ascending=False)

print(f"\nBiggest prediction errors (model was most wrong):")
display(results_df.head(10))

Shows available for modeling: 198
Feature matrix: (198, 32)
Train: 158 shows | Test: 40 shows

Linear Regression:
  MAE : 0.0235  (avg error in engagement score units)
  R²  : 0.9778  (1.0 = perfect, 0 = no better than mean)

Top 10 features that predict HIGH engagement:


,feature,coefficient
6,completion_ratio,0.509201
7,binge_score,0.250125
8,pace_norm,0.233855
24,genre_Drama,0.014795
30,genre_Soap,0.013282
1,darkness,0.009631
3,emotional_intensity,0.002453
28,genre_Reality,0.002344
10,tone_humorous,0.002010
27,genre_Mystery,0.001605



Top 10 features that predict LOW engagement:


,feature,coefficient
21,genre_Comedy,-0.012175
25,genre_Family,-0.013492
0,pacing,-0.013569
15,tone_philosophical,-0.022795
26,genre_Kids,-0.022946
9,tone_tense,-0.025966
11,tone_romantic,-0.029954
17,tone_whimsical,-0.030093
14,tone_nostalgic,-0.030933
18,tone_gritty,-0.036789



Biggest prediction errors (model was most wrong):


,show,actual,predicted,error
57,The OA,0.7300,0.9043,0.1743
56,התאמה מושלמת,0.7333,0.9027,0.1694
18,דברים מוזרים,0.8550,0.8068,-0.0482
186,הסודות של קורון,0.1856,0.2267,0.0411
24,השבטים,0.8417,0.8026,-0.0391
147,הוליווד,0.2561,0.2194,-0.0367
197,הפלא,0.1525,0.1174,-0.0351
75,הרחק,0.6414,0.6703,0.0289
223,שאלת מיליון הדולר,0.0867,0.0602,-0.0265
16,מטורף,0.8670,0.8917,0.0247


In [44]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# ── Content-only features (no behavioral data) ────────────────────────────────
content_feature_cols = thematic_cols + tone_cols + genre_cols

X_content = pd.concat([
    model_df[thematic_cols + tone_cols],
    genre_df_ohe
], axis=1)
y = model_df['engagement_score']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_content, y, test_size=0.2, random_state=42
)

print(f"Content-only feature matrix: {X_content.shape}")
print(f"Train: {len(X_train_c)} | Test: {len(X_test_c)}")

# ── Model 1: Linear Regression ────────────────────────────────────────────────
lr_c = LinearRegression()
lr_c.fit(X_train_c, y_train_c)
y_pred_lr_c = lr_c.predict(X_test_c)

mae_lr_c = mean_absolute_error(y_test_c, y_pred_lr_c)
r2_lr_c  = r2_score(y_test_c, y_pred_lr_c)

# ── Model 2: Ridge Regression (handles correlated features better) ────────────
ridge_c = Ridge(alpha=1.0)
ridge_c.fit(X_train_c, y_train_c)
y_pred_ridge_c = ridge_c.predict(X_test_c)

mae_ridge_c = mean_absolute_error(y_test_c, y_pred_ridge_c)
r2_ridge_c  = r2_score(y_test_c, y_pred_ridge_c)

# ── Model 3: Random Forest ────────────────────────────────────────────────────
rf_c = RandomForestRegressor(n_estimators=100, random_state=42)
rf_c.fit(X_train_c, y_train_c)
y_pred_rf_c = rf_c.predict(X_test_c)

mae_rf_c = mean_absolute_error(y_test_c, y_pred_rf_c)
r2_rf_c  = r2_score(y_test_c, y_pred_rf_c)

# ── Compare all models ────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model':     ['Linear Regression', 'Ridge Regression', 'Random Forest'],
    'MAE':       [mae_lr_c, mae_ridge_c, mae_rf_c],
    'R²':        [r2_lr_c, r2_ridge_c, r2_rf_c],
    'Features':  ['content only'] * 3
}).round(4)

print("\n── Content-only models ──────────────────────────────────────")
display(comparison)

# ── Feature importance from Random Forest ────────────────────────────────────
rf_importance = pd.DataFrame({
    'feature':    X_content.columns,
    'importance': rf_c.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 most important features (Random Forest):")
display(rf_importance.head(15).round(4))

# ── Predicted vs Actual for best model ───────────────────────────────────────
best_pred = y_pred_ridge_c  # will update after seeing results
results_content = pd.DataFrame({
    'show':      model_df.loc[X_test_c.index, 'show'],
    'actual':    y_test_c.values,
    'predicted': best_pred.round(4),
    'error':     (best_pred - y_test_c.values).round(4)
}).sort_values('error', key=abs, ascending=False)

print("\nBiggest prediction errors (content-only):")
display(results_content.head(10))

# ── Cross-validation for more robust evaluation ───────────────────────────────
# 5-fold CV gives a more reliable estimate than single 80/20 split
cv_scores_ridge = cross_val_score(Ridge(alpha=1.0), X_content, y,
                                   cv=5, scoring='r2')
cv_scores_rf    = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42),
                                   X_content, y, cv=5, scoring='r2')

print("\n── Cross-validation R² (5-fold) ─────────────────────────────")
print(f"Ridge:         {cv_scores_ridge.mean():.4f} ± {cv_scores_ridge.std():.4f}")
print(f"Random Forest: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")

Content-only feature matrix: (198, 29)
Train: 158 | Test: 40

── Content-only models ──────────────────────────────────────


,Model,MAE,R²,Features
0,Linear Regression,0.2393,0.0501,content only
1,Ridge Regression,0.2272,0.1747,content only
2,Random Forest,0.2303,0.1414,content only



Top 15 most important features (Random Forest):


,feature,importance
1,darkness,0.1067
4,mystery_level,0.1038
5,realism,0.0947
0,pacing,0.0634
9,tone_disturbing,0.0574
12,tone_philosophical,0.0565
2,complexity,0.0531
3,emotional_intensity,0.0490
24,genre_Mystery,0.0474
6,tone_tense,0.0377



Biggest prediction errors (content-only):


,show,actual,predicted,error
205,מבפנים,0.1312,0.7780,0.6468
24,השבטים,0.8417,0.3115,-0.5302
203,מוגנים,0.1332,0.5845,0.4513
239,ספרטקוס,0.0727,0.4746,0.4019
186,הסודות של קורון,0.1856,0.5698,0.3842
19,המתנקשים,0.8549,0.5038,-0.3511
127,בובה רוסית,0.3245,0.6739,0.3494
16,מטורף,0.8670,0.5341,-0.3329
9,חינוך מיני,0.8986,0.6174,-0.2812
123,נפשות תאומות,0.3297,0.5907,0.2610



── Cross-validation R² (5-fold) ─────────────────────────────
Ridge:         -151.5537 ± 224.1309
Random Forest: -164.1677 ± 248.9074


I'll train a streamlined content-only model using fewer, more high-level features to test if reducing dimensionality eliminates overfitting and improves generalizability

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Load from saved files ─────────────────────────────────────────────────────
show_df    = pd.read_csv('show_df.csv')
show_df['genres_list'] = show_df['genres_list'].apply(eval)
show_df['tone_tags']   = show_df['tone_tags'].apply(eval)

thematic_cols = ['pacing', 'darkness', 'complexity',
                 'emotional_intensity', 'mystery_level', 'realism']
all_tone_tags = ['tense','humorous','romantic','disturbing',
                 'inspiring','nostalgic','philosophical',
                 'suspenseful','whimsical','gritty']

# Rebuild model_df
model_df = show_df.dropna(subset=['completion_ratio'] + thematic_cols).copy()
for tag in all_tone_tags:
    model_df[f'tone_{tag}'] = model_df['tone_tags'].apply(
        lambda x: 1 if isinstance(x, list) and tag in x else 0
    )
tone_cols = [f'tone_{t}' for t in all_tone_tags]

# Rebuild genre matrix
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(model_df['genres_list'])
genre_cols   = [f"genre_{g.replace(' ','_').replace('&','and').replace('-','_')}"
                for g in mlb.classes_]
genre_df_ohe = pd.DataFrame(genre_matrix, columns=genre_cols, index=model_df.index)

print(f"Loaded: {len(show_df)} shows | model_df: {len(model_df)} shows")

# ── Previous 29-feature model (for comparison) ────────────────────────────────
feature_cols_29 = thematic_cols + tone_cols
X_29 = pd.concat([model_df[feature_cols_29], genre_df_ohe], axis=1)
y    = model_df['engagement_score']

X_train_29, X_test_29, y_train_29, y_test_29 = train_test_split(
    X_29, y, test_size=0.2, random_state=42
)
ridge_29 = Ridge(alpha=1.0).fit(X_train_29, y_train_29)
rf_29    = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1).fit(X_train_29, y_train_29)

mae_ridge_29 = mean_absolute_error(y_test_29, ridge_29.predict(X_test_29))
r2_ridge_29  = r2_score(y_test_29, ridge_29.predict(X_test_29))
mae_rf_29    = mean_absolute_error(y_test_29, rf_29.predict(X_test_29))
r2_rf_29     = r2_score(y_test_29, rf_29.predict(X_test_29))

# ── 6 Thematic features only ──────────────────────────────────────────────────
X_6 = model_df[thematic_cols]

X_train_6, X_test_6, y_train_6, y_test_6 = train_test_split(
    X_6, y, test_size=0.2, random_state=42
)
ridge_6 = Ridge(alpha=1.0).fit(X_train_6, y_train_6)
rf_6    = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1).fit(X_train_6, y_train_6)

mae_ridge_6 = mean_absolute_error(y_test_6, ridge_6.predict(X_test_6))
r2_ridge_6  = r2_score(y_test_6, ridge_6.predict(X_test_6))
mae_rf_6    = mean_absolute_error(y_test_6, rf_6.predict(X_test_6))
r2_rf_6     = r2_score(y_test_6, rf_6.predict(X_test_6))

# ── Comparison table ──────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model':    ['Ridge (29 features)', 'Ridge (6 thematic only)',
                 'Random Forest (29)', 'Random Forest (6 thematic only)'],
    'MAE':      [mae_ridge_29, mae_ridge_6, mae_rf_29, mae_rf_6],
    'R²':       [r2_ridge_29,  r2_ridge_6,  r2_rf_29,  r2_rf_6],
}).round(4)

print("\n── Content-only models: 29 features vs 6 thematic ──────────")
display(comparison)

# ── Ridge coefficients for 6-feature model ───────────────────────────────────
ridge_coef = pd.DataFrame({
    'feature':     thematic_cols,
    'coefficient': ridge_6.coef_
}).sort_values('coefficient', ascending=False)

print("\nThematic feature impact (Ridge coefficients):")
display(ridge_coef.round(4))

# ── Cross-validation Ridge only ───────────────────────────────────────────────
cv_6  = cross_val_score(Ridge(alpha=1.0), X_6,  y, cv=5, scoring='r2')
cv_29 = cross_val_score(Ridge(alpha=1.0), X_29, y, cv=5, scoring='r2')

print(f"\nCross-validation R² (5-fold):")
print(f"  Ridge 29 features:  {cv_29.mean():.4f} ± {cv_29.std():.4f}")
print(f"  Ridge 6 thematic:   {cv_6.mean():.4f}  ± {cv_6.std():.4f}")

Loaded: 246 shows | model_df: 198 shows

── Content-only models: 29 features vs 6 thematic ──────────


,Model,MAE,R²
0,Ridge (29 features),0.2217,0.2620
1,Ridge (6 thematic only),0.2461,0.1254
2,Random Forest (29),0.2527,-0.0104
3,Random Forest (6 thematic only),0.2334,0.1264



Thematic feature impact (Ridge coefficients):


,feature,coefficient
3,emotional_intensity,0.0679
4,mystery_level,0.0293
1,darkness,0.0132
2,complexity,0.0049
5,realism,-0.0166
0,pacing,-0.0390



Cross-validation R² (5-fold):
  Ridge 29 features:  -153.5653 ± 226.3599
  Ridge 6 thematic:   -160.7962  ± 241.6080


The results show that user taste is highly non-linear, as shows with similar thematic profiles received wildly different scores, preventing the model from finding a reliable content-based prediction pattern

To build a truly effective predictive model without causing data leakage, we would need external, aggregated user behavior data—such as global completion rates and watching trends for each show. This would capture the collective engagement of the audience as a powerful predictive feature, rather than relying solely on the show's static content metrics

Let's train similarity model and exam the output:

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

# ── Build thematic vectors for all shows ─────────────────────────────────────
thematic_cols = ['pacing', 'darkness', 'complexity',
                 'emotional_intensity', 'mystery_level', 'realism']

sim_df = model_df.dropna(subset=thematic_cols).copy()

# Normalize each dimension to 0-1 (they're already 1-5, so divide by 5)
thematic_matrix = sim_df[thematic_cols].values / 5.0

# Cosine similarity between all shows
similarity_matrix = cosine_similarity(thematic_matrix)
sim_matrix_df = pd.DataFrame(
    similarity_matrix,
    index=sim_df['show'],
    columns=sim_df['show']
)

# ── Find shows most similar to your top-engaged shows ────────────────────────
def get_similar_shows(show_name, top_n=5, min_engagement=None):
    if show_name not in sim_matrix_df.index:
        print(f"'{show_name}' not found")
        return None
    
    similarities = sim_matrix_df[show_name].drop(show_name).sort_values(ascending=False)
    
    result = pd.DataFrame({
        'show':       similarities.index,
        'similarity': similarities.values,
        'engagement': sim_df.set_index('show').loc[similarities.index, 'engagement_score'].values
    })
    
    if min_engagement is not None:
        result = result[result['engagement'] >= min_engagement]
    
    return result.head(top_n)

# Test on your top shows
for show in ['המכשף', 'מיינד האנטר', 'בעיית שלושת הגופים', 'Dark']:
    if show in sim_matrix_df.index:
        print(f"\nShows most similar to '{show}':")
        display(get_similar_shows(show, top_n=5))

# ── Predict engagement for unseen shows using similarity ─────────────────────
# For each show: predicted_engagement = weighted average of similar shows' engagement
# weighted by cosine similarity score

def predict_engagement_by_similarity(show_name, top_n=10):
    if show_name not in sim_matrix_df.index:
        return None
    
    similar = get_similar_shows(show_name, top_n=top_n)
    if similar is None:
        return None
    
    # Weighted average: more similar shows have more influence
    weights  = similar['similarity'].values
    eng      = similar['engagement'].values
    predicted = np.average(eng, weights=weights)
    return round(predicted, 4)

# Evaluate on test set
test_shows_list = sim_df.loc[X_test_c.index, 'show'].values
predictions, actuals, show_names = [], [], []

for show in test_shows_list:
    pred = predict_engagement_by_similarity(show, top_n=10)
    if pred is not None:
        actual = sim_df.set_index('show').loc[show, 'engagement_score']
        predictions.append(pred)
        actuals.append(actual)
        show_names.append(show)

mae_sim  = mean_absolute_error(actuals, predictions)
r2_sim   = r2_score(actuals, predictions)

print(f"\n── Similarity Engine performance ────────────────────────────")
print(f"MAE : {mae_sim:.4f}")
print(f"R²  : {r2_sim:.4f}")

results_sim = pd.DataFrame({
    'show':      show_names,
    'actual':    actuals,
    'predicted': predictions,
    'error':     [p - a for p, a in zip(predictions, actuals)]
}).sort_values('error', key=abs, ascending=False)

print(f"\nBiggest errors:")
display(results_sim.head(10))

# ── Final model comparison ────────────────────────────────────────────────────
print(f"\n── Model comparison (content-only, no leakage) ──────────────")
comparison = pd.DataFrame({
    'Model':   ['Linear Regression', 'Ridge Regression',
                'Random Forest',     'Similarity Engine'],
    'MAE':     [mae_lr_c, mae_ridge_c, mae_rf_c, mae_sim],
    'R²':      [r2_lr_c,  r2_ridge_c,  r2_rf_c,  r2_sim],
}).round(4)
display(comparison)


Shows most similar to 'המכשף':


,show,similarity,engagement
0,מטורף,1.000000,0.8670
1,מגוף לגוף,0.995638,0.6961
2,פחמן משודרג,0.995638,0.8083
3,ברוכים הבאים לעדן,0.995638,0.7986
4,The OA,0.995638,0.7300



Shows most similar to 'מיינד האנטר':


,show,similarity,engagement
0,לא ייאמן,0.996419,0.8469
1,אסונטה,0.996419,0.7858
2,התבגרות,0.996419,0.7850
3,מחלקה Q,0.995806,0.8700
4,תישאר קרוב,0.995806,0.9100



Shows most similar to 'בעיית שלושת הגופים':


,show,similarity,engagement
0,משחקים גרייסי דרלינג,0.995871,0.2905
1,אקווינוקס,0.995871,0.0976
2,אוסמוסיס,0.993042,0.3463
3,ראגנארוק,0.992062,0.5980
4,ארכיון 81,0.992062,0.8250



── Similarity Engine performance ────────────────────────────
MAE : 0.2194
R²  : 0.1985

Biggest errors:


,show,actual,predicted,error
10,מבפנים,0.1312,0.6360,0.5048
23,רובוטים,0.0767,0.5560,0.4793
37,יחידה 42,0.0980,0.5723,0.4743
17,חינוך מיני,0.8986,0.4268,-0.4718
4,מוגנים,0.1332,0.5234,0.3902
38,המתנקשים,0.8549,0.4708,-0.3841
6,הסודות של קורון,0.1856,0.5427,0.3571
20,קסנדרה,0.3395,0.6876,0.3481
5,זרה,0.2516,0.5895,0.3379
3,זאב צעיר,0.1772,0.5064,0.3292



── Model comparison (content-only, no leakage) ──────────────


,Model,MAE,R²
0,Linear Regression,0.2393,0.0501
1,Ridge Regression,0.2272,0.1747
2,Random Forest,0.2303,0.1414
3,Similarity Engine,0.2194,0.1985
